### 0. Training Checkpoints & Disconnection Protection
To prevent losing training progress due to disconnections, we save checkpoints regularly to the local disk during training. You can download these checkpoints or use the final saved adapter model `qwen-vl-phishing-lora-final` directly.

In [ ]:
import os

# Create a directory for checkpoints
CHECKPOINT_DIR = "/content/qwen2.5-vl-phishing-lora-checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Local checkpoint directory ready at: {CHECKPOINT_DIR}")

Local checkpoint directory ready at: /content/qwen2.5-vl-phishing-lora-checkpoints


### 1. Load Datasets
download the raw phishing screenshots dataset.

In [ ]:
import os
from google.colab import userdata

# Securely fetch Kaggle credentials from Colab Secrets
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("Kaggle credentials successfully loaded from secrets.")
except userdata.SecretNotFoundError:
    print("WARNING: Kaggle credentials not found in secrets! Please add KAGGLE_USERNAME and KAGGLE_KEY to the Colab Secrets panel.")

# Install Kaggle library
!pip install -q kaggle

# Download and unzip the dataset
!kaggle datasets download -d zackyzac/phishing-sites-screenshot --unzip -p /content/kaggle_phishing_dataset

print("Dataset downloaded to /content/kaggle_phishing_dataset")

Kaggle credentials successfully loaded from secrets.
Dataset URL: https://www.kaggle.com/datasets/zackyzac/phishing-sites-screenshot
License(s): MIT
100% 201M/201M [00:11<00:00, 18.4MB/s]

Dataset downloaded to /content/kaggle_phishing_dataset


In [ ]:
import os

KAGGLE_DATASET_DIR = "/content/kaggle_phishing_dataset"

# Let's inspect the contents to see how to build our Chain-of-Thought JSON
if os.path.exists(KAGGLE_DATASET_DIR):
    # Walk through the directory to see the structure
    for root, dirs, files in os.walk(KAGGLE_DATASET_DIR):
        if files:
            print(f"Found {len(files)} files in {root}")
            print(f"Sample files: {files[:3]}\n")
else:
    print("Dataset directory not found. Please ensure the download completed successfully.")

Found 1147 files in /content/kaggle_phishing_dataset/screenshots/genuine_site_0
Sample files: ['genuine_amazon.comAlex-Tothdp0878163719_2377.png', 'genuine_www.devx.comcplus_352.png', 'genuine_play.tmstory1666_2199.png']

Found 550 files in /content/kaggle_phishing_dataset/screenshots/phishing_site_1
Sample files: ['phishing_www.bjcurio.comjsindex.htm_2933.png', 'phishing_xanhtuoi.comindex.htm_2852.png', 'phishing_johnsondistribuidores.comadmin5GDindex.php_1967.png']



#### Format Data for Vision Model
We convert the raw images into a structured JSONL format containing the conversational prompts required by Qwen2.5-VL.

In [ ]:
import glob
import json
import os
import random

KAGGLE_DATASET_DIR = "/content/kaggle_phishing_dataset/screenshots"
train_metadata = []

# Define paths
phishing_dir = os.path.join(KAGGLE_DATASET_DIR, "phishing_site_1")
genuine_dir = os.path.join(KAGGLE_DATASET_DIR, "genuine_site_0")

def process_images(directory, is_phishing):
    images = glob.glob(os.path.join(directory, "*.png")) + glob.glob(os.path.join(directory, "*.jpg"))
    for img_path in images:

        # Mocking target brand and cues since the raw dataset doesn't have detailed annotations
        target_brand = "Unknown"
        cues = ["suspicious URL or layout"] if is_phishing else []

        assistant_target = {
            "target_brand": target_brand,
            "cues": cues,
            "is_phishing": is_phishing
        }

        train_metadata.append({
            "image_path": img_path,
            "conversations": [
                {
                    "from": "user",
                    "value": f"Analyze this webpage screenshot. Identify the target brand, list suspicious visual or textual cues, and classify whether it is phishing in JSON format."
                },
                {
                    "from": "assistant",
                    "value": json.dumps(assistant_target)
                }
            ]
        })

# Process both directories
process_images(phishing_dir, is_phishing=True)
process_images(genuine_dir, is_phishing=False)

# Shuffle the dataset
random.shuffle(train_metadata)

# Save the formatted dataset
output_jsonl = "/content/train_metadata.jsonl"
with open(output_jsonl, "w") as f:
    for entry in train_metadata:
        f.write(json.dumps(entry) + "\n")

print(f"Successfully generated {len(train_metadata)} formatted training examples at {output_jsonl}")

# Display a sample
print("\nSample entry:")
print(json.dumps(train_metadata[0], indent=2))

Successfully generated 1697 formatted training examples at /content/train_metadata.jsonl

Sample entry:
{
  "image_path": "/content/kaggle_phishing_dataset/screenshots/phishing_site_1/phishing_sh4d0wxpl01tr.hoxty.com_2669.png",
  "conversations": [
    {
      "from": "user",
      "value": "Analyze this webpage screenshot. Identify the target brand, list suspicious visual or textual cues, and classify whether it is phishing in JSON format."
    },
    {
      "from": "assistant",
      "value": "{\"target_brand\": \"Unknown\", \"cues\": [\"suspicious URL or layout\"], \"is_phishing\": true}"
    }
  ]
}


In [ ]:
import glob
import json
import os
import random
import re

KAGGLE_DATASET_DIR = "/content/kaggle_phishing_dataset/screenshots"
train_metadata = []

phishing_dir = os.path.join(KAGGLE_DATASET_DIR, "phishing_site_1")
genuine_dir = os.path.join(KAGGLE_DATASET_DIR, "genuine_site_0")

def clean_brand_name(filename):
    # Extract domain name as the brand from names like genuine_linkedin.comxxx.png or phishing_www.tuhu.cc.png
    name = os.path.basename(filename)
    # Strip prefixes
    name = re.sub(r'^(genuine_|phishing_)', '', name)
    # Strip common subdomains
    name = re.sub(r'^(www\.)', '', name)
    # Extract up to the first top-level domain or separator
    match = re.match(r'^([^._\s/]+)', name)
    if match:
        brand = match.group(1)
        # Capitalize first letter for display
        return brand.capitalize()
    return "Unknown"

def process_images_improved(directory, is_phishing):
    images = glob.glob(os.path.join(directory, "*.png")) + glob.glob(os.path.join(directory, "*.jpg"))
    for img_path in images:
        target_brand = clean_brand_name(img_path)

        if is_phishing:
            cues = [
                f"The page mimics {target_brand} but has a suspicious or deceptive URL structure.",
                "Misaligned layouts or input fields asking for sensitive user credentials."
            ]
        else:
            cues = []

        assistant_target = {
            "target_brand": target_brand,
            "cues": cues,
            "is_phishing": is_phishing
        }

        train_metadata.append({
            "image_path": img_path,
            "conversations": [
                {
                    "from": "user",
                    "value": "Analyze this webpage screenshot. Identify the target brand, list suspicious visual or textual cues, and classify whether it is phishing in JSON format."
                },
                {
                    "from": "assistant",
                    "value": json.dumps(assistant_target)
                }
            ]
        })

# Re-process both directories with brand extraction enabled
process_images_improved(phishing_dir, is_phishing=True)
process_images_improved(genuine_dir, is_phishing=False)

# Shuffle and save the newly enriched dataset
random.shuffle(train_metadata)
output_jsonl = "/content/train_metadata.jsonl"
with open(output_jsonl, "w") as f:
    for entry in train_metadata:
        f.write(json.dumps(entry) + "\n")

print(f"Successfully generated {len(train_metadata)} enriched training examples at {output_jsonl}!")
print("\nSample entry with extracted brand name:")
print(json.dumps(train_metadata[0], indent=2))

Successfully generated 1697 enriched training examples at /content/train_metadata.jsonl!

Sample entry with extracted brand name:
{
  "image_path": "/content/kaggle_phishing_dataset/screenshots/phishing_site_1/phishing_eventplus.co.inlogin1secured.paypalwebscr.php_1892.png",
  "conversations": [
    {
      "from": "user",
      "value": "Analyze this webpage screenshot. Identify the target brand, list suspicious visual or textual cues, and classify whether it is phishing in JSON format."
    },
    {
      "from": "assistant",
      "value": "{\"target_brand\": \"Eventplus\", \"cues\": [\"The page mimics Eventplus but has a suspicious or deceptive URL structure.\", \"Misaligned layouts or input fields asking for sensitive user credentials.\"], \"is_phishing\": true}"
    }
  ]
}


In [ ]:
import json
from sklearn.model_selection import train_test_split

# Helper to save JSONL
def save_jsonl(dataset, path):
    with open(path, "w") as f:
        for entry in dataset:
            f.write(json.dumps(entry) + "\n")

# Re-split the dataset using our enriched labels
with open("/content/train_metadata.jsonl", "r") as f:
    data = [json.loads(line) for line in f]

train_data, temp_data = train_test_split(data, test_size=0.2, random_state=42)
valid_size = min(150, len(temp_data) // 2)
valid_data = temp_data[:valid_size]
test_data = temp_data[valid_size:]

save_jsonl(train_data, "/content/train.jsonl")
save_jsonl(valid_data, "/content/valid.jsonl")
save_jsonl(test_data, "/content/test.jsonl")

print("Re-split enriched datasets successfully saved to local files!")

Re-split enriched datasets successfully saved to local files!


#### Install Dependencies
Next, we install Unsloth and other required ML libraries to handle training and quantization efficiently.

In [ ]:
# Install required ML libraries
!pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U "transformers<=5.17.0" accelerate peft "trl<=1.15.0" bitsandbytes qwen-vl-utils scikit-learn

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


#### Split Dataset
We split the formatted data into training (80%), validation (for potential advanced quantization), and test sets.

In [ ]:
import json
import random
import os
from sklearn.model_selection import train_test_split

# Safety check to ensure previous cells were run
if not os.path.exists("/content/train_metadata.jsonl"):
    raise FileNotFoundError("Dataset not found! Please run the first few cells to download and generate train_metadata.jsonl")

# Load the formatted dataset
with open("/content/train_metadata.jsonl", "r") as f:
    data = [json.loads(line) for line in f]

# Split into Train (80%) and Temp (20%)
train_data, temp_data = train_test_split(data, test_size=0.2, random_state=42)

# Split Temp into Test and Validation (~150 samples for Validation)
valid_size = min(150, len(temp_data) // 2)
valid_data = temp_data[:valid_size]
test_data = temp_data[valid_size:]

# Helper to save JSONL
def save_jsonl(dataset, path):
    with open(path, "w") as f:
        for entry in dataset:
            f.write(json.dumps(entry) + "\n")

save_jsonl(train_data, "/content/train.jsonl")
save_jsonl(valid_data, "/content/valid.jsonl")
save_jsonl(test_data, "/content/test.jsonl")

print("Dataset splitting complete!")
print(f"Train Set: {len(train_data)} samples")
print(f"Validation Set: {len(valid_data)} samples")
print(f"Test Set: {len(test_data)} samples")

Dataset splitting complete!
Train Set: 1357 samples
Validation Set: 150 samples
Test Set: 190 samples


### 2. Raw Model Baseline
Let's load the unquantized (16-bit BF16) baseline model to see how it performs out of the box.

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

print("Downloading and loading Qwen2.5-VL-7B-Instruct (this may take a few minutes)...")

model_id = "Qwen/Qwen2.5-VL-7B-Instruct"

# Load processor
processor = AutoProcessor.from_pretrained(model_id)

# Load model in bfloat16 (unquantized baseline)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("Model successfully loaded into VRAM in BF16 format!")

W1112 15:13:20.785000 993 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 15:13:20.835000 993 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

... (4 outputs left out)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Model successfully loaded into VRAM in BF16 format!


#### Evaluate Baseline
We evaluate the raw model on a small subset of our test data to establish baseline accuracy and latency metrics.

In [ ]:
import json
import torch
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from qwen_vl_utils import process_vision_info

# Load test data
with open("/content/test.jsonl", "r") as f:
    test_data = [json.loads(line) for line in f]

true_labels = []
pred_labels = []
latencies = []

# To save time in this notebook, we'll evaluate on a subset of 20 images
# For a full benchmark, you would use the entire test_data
test_subset = test_data[:20]
print(f"Evaluating baseline on {len(test_subset)} test samples...")

for i, sample in enumerate(test_subset):
    user_msg = sample["conversations"][0]["value"]
    img_path = sample["image_path"]

    # The true target is in the assistant's message, stored as a JSON string
    true_target = json.loads(sample["conversations"][1]["value"])
    true_labels.append(true_target["is_phishing"])

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": user_msg},
            ],
        }
    ]

    # Format inputs for Qwen2.5-VL
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    # Generate prediction
    start_time = time.time()
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.0, # Greedy decoding for deterministic results
            do_sample=False
        )
    latency = time.time() - start_time
    latencies.append(latency)

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    # Attempt to parse the model's JSON output
    try:
        start_idx = output_text.find('{')
        end_idx = output_text.rfind('}') + 1
        json_str = output_text[start_idx:end_idx]
        pred_target = json.loads(json_str)
        pred_labels.append(pred_target.get("is_phishing", False))
    except Exception as e:
        # If parsing fails or the model hallucinates bad JSON, default to False
        pred_labels.append(False)

    if (i + 1) % 5 == 0:
        print(f"Processed {i + 1}/{len(test_subset)}...")

# Compute Metrics
accuracy = accuracy_score(true_labels, pred_labels)
precision = precision_score(true_labels, pred_labels, zero_division=0)
recall = recall_score(true_labels, pred_labels, zero_division=0)
f1 = f1_score(true_labels, pred_labels, zero_division=0)

# False Negative Rate (Very important for security: catching phishing sites)
fn = sum((t == True and p == False) for t, p in zip(true_labels, pred_labels))
total_true_phishing = sum(true_labels)
fnr = fn / total_true_phishing if total_true_phishing > 0 else 0

vram_peak = torch.cuda.max_memory_allocated() / (1024**3)
avg_latency = sum(latencies) / len(latencies)

print("\n" + "="*40)
print(f"--- Baseline Metrics (Unquantized BF16) ---")
print("="*40)
print(f"Accuracy:             {accuracy:.4f}")
print(f"Precision:            {precision:.4f}")
print(f"Recall:               {recall:.4f}")
print(f"F1-Score:             {f1:.4f}")
print(f"False Negative Rate:  {fnr:.4f}")
print(f"VRAM Peak:            {vram_peak:.2f} GB")
print(f"Average Latency:      {avg_latency:.2f} sec/image")
print("="*40)

Evaluating baseline on 20 test samples...


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Processed 5/20...
Processed 10/20...
Processed 15/20...
Processed 20/20...

--- Baseline Metrics (Unquantized BF16) ---
Accuracy:             0.5500
Precision:            0.0000
Recall:               0.0000
F1-Score:             0.0000
False Negative Rate:  1.0000
VRAM Peak:            15.72 GB
Average Latency:      4.12 sec/image


In [ ]:
import gc
import torch

# 1. Free up VRAM from the baseline model
print("Clearing baseline model from VRAM...")
if 'model' in globals():
    del model
if 'processor' in globals():
    del processor

torch.cuda.empty_cache()
gc.collect()

print("VRAM cleared! Ready for QLoRA.")

Clearing baseline model from VRAM...
VRAM cleared! Ready for QLoRA.


### 3. Quantize & Train (LoRA 4-bit)
We load the base model in 4-bit precision using Unsloth and apply LoRA adapters to prepare for memory-efficient fine-tuning.

In [ ]:
from unsloth import FastVisionModel

print("Loading Qwen2.5-VL-7B-Instruct in 4-bit using Unsloth...")
model, processor = FastVisionModel.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    load_in_4bit = True, # Use 4-bit to reduce memory footprint
    use_gradient_checkpointing = "unsloth", # Saves more VRAM during training
)

# Apply LoRA adapters
print("Applying LoRA adapters...")
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers = False, # We only fine-tune the language model part to save memory and speed up
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = 16, # Rank of the LoRA matrices
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("Model and adapters ready for training!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:2161: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Qwen2.5-VL-7B-Instruct in 4-bit using Unsloth...
==((====))==  Unsloth 2026.11.4: Fast Qwen2_5_Vl patching. Transformers: 5.17.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen2_5_VLDecoderLayer).
Applying LoRA adapters...
Model and adapters ready for training!


#### Prepare Training Data
We format the previously split `train.jsonl` into the exact dataset format expected by Unsloth and the Hugging Face Trainer.

In [ ]:
from datasets import load_dataset
import warnings

# Ignore warnings
warnings.filterwarnings('ignore')

print("Formatting dataset for Unsloth and TRL...")
dataset = load_dataset("json", data_files="/content/train.jsonl", split="train")

def format_for_vision(example):
    # Pass the local file path directly instead of a PIL Image
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": example["image_path"]},
                    {"type": "text", "text": example["conversations"][0]["value"]}
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": example["conversations"][1]["value"]}
                ]
            }
        ]
    }

# Map the formatting function.
train_dataset = dataset.map(format_for_vision, remove_columns=dataset.column_names, num_proc=4)
print(f"Prepared {len(train_dataset)} training examples.")

Formatting dataset for Unsloth and TRL...


Generating train split: 0 examples [00:00, ? examples/s]

Map (num_proc=4):   0%|          | 0/1357 [00:00<?, ? examples/s]

Prepared 1357 training examples.


#### Run Fine-Tuning
Now we initialize the `SFTTrainer` and execute the fine-tuning process. This step updates the LoRA adapters while keeping the base 4-bit model frozen.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported
from unsloth import UnslothVisionDataCollator
from datasets import load_dataset
import warnings

warnings.filterwarnings('ignore')

# Re-load our freshly generated enriched dataset
print("Loading the newly brand-enriched dataset...")
enriched_dataset = load_dataset("json", data_files="/content/train.jsonl", split="train")

def format_for_vision(example):
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": example["image_path"]},
                    {"type": "text", "text": example["conversations"][0]["value"]}
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": example["conversations"][1]["value"]}
                ]
            }
        ]
    }

train_dataset_enriched = enriched_dataset.map(format_for_vision, remove_columns=enriched_dataset.column_names, num_proc=4)
print(f"Prepared {len(train_dataset_enriched)} enriched training examples.")

# Prepare the model for gradient calculation
FastVisionModel.for_training(model)

print("Initializing SFTTrainer on brand-enriched dataset...")
trainer = SFTTrainer(
    model = model,
    tokenizer = processor,
    data_collator = UnslothVisionDataCollator(model, processor),
    train_dataset = train_dataset_enriched,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "qwen2.5-vl-phishing-lora",
        report_to = "none",
        remove_unused_columns = False,
        dataset_kwargs = {"skip_prepare_dataset": True},
        dataset_num_proc = 4,
        max_seq_length = 2048,
    ),
)

print("Starting fine-tuning with unbiased labels...")
trainer_stats = trainer.train()
print("Training Complete!")

### 4. Save the Fine-Tuned Adapters
The fine-tuning is now complete. We will save the model first before loading it in different quantization levels (16-bit, 8-bit, 4-bit) for a fair comparison.

In [ ]:
print("Model training is complete. Proceeding to save the model and evaluate multiple quantization levels.")

Model training is complete. Proceeding to save the model and evaluate multiple quantization levels.


### Save the Fine-Tuned Model
Let's save the LoRA adapters and the processor so you can easily load them later for inference without retraining.

In [ ]:
# Save the LoRA adapters locally
save_directory = "/content/qwen-vl-phishing-lora-final"

print(f"Saving model adapters to {save_directory}...")
model.save_pretrained(save_directory)
processor.save_pretrained(save_directory)

print("Model saved successfully!")

# NOTE: If you want to push this to Hugging Face, you can uncomment the lines below and log in.
# from huggingface_hub import login
# login(token="YOUR_HF_WRITE_TOKEN")
# model.push_to_hub("your-username/qwen-vl-phishing-lora")
# processor.push_to_hub("your-username/qwen-vl-phishing-lora")

Saving model adapters to /content/qwen-vl-phishing-lora-final...


Unsloth: Restored added_tokens_decoder metadata in /content/qwen-vl-phishing-lora-final/tokenizer_config.json.


Model saved successfully!


### 5. Quantization Comparison
Let's modularize our evaluation function so we can easily test the model at different quantization levels (16-bit, 8-bit, and 4-bit).

In [ ]:
import gc
import time
import json
import torch
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from qwen_vl_utils import process_vision_info

def evaluate_quantization(eval_model, eval_processor, subset):
    if hasattr(eval_model, "for_inference"):
        eval_model.for_inference()
    else:
        eval_model.eval()

    pred_labels = []
    latencies = []
    true_labels = []

    for i, sample in enumerate(subset):
        user_msg = sample["conversations"][0]["value"]
        img_path = sample["image_path"]

        true_target = json.loads(sample["conversations"][1]["value"])
        true_labels.append(true_target["is_phishing"])

        messages = [{"role": "user", "content": [{"type": "image", "image": img_path}, {"type": "text", "text": user_msg}]}]

        text = eval_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = eval_processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to("cuda")

        start_time = time.time()
        with torch.no_grad():
            generated_ids = eval_model.generate(**inputs, max_new_tokens=128, temperature=0.0, do_sample=False)
        latencies.append(time.time() - start_time)

        generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
        output_text = eval_processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

        try:
            start_idx = output_text.find('{')
            end_idx = output_text.rfind('}') + 1
            pred_target = json.loads(output_text[start_idx:end_idx])
            pred_labels.append(pred_target.get("is_phishing", False))
        except:
            pred_labels.append(False)

        if (i + 1) % 5 == 0:
            print(f"Processed {i + 1}/{len(subset)}...")

    total_phish = sum(true_labels)
    fn = sum((t == True and p == False) for t, p in zip(true_labels, pred_labels))

    return {
        "Accuracy": accuracy_score(true_labels, pred_labels),
        "Precision": precision_score(true_labels, pred_labels, zero_division=0),
        "Recall": recall_score(true_labels, pred_labels, zero_division=0),
        "F1-Score": f1_score(true_labels, pred_labels, zero_division=0),
        "FNR": fn / total_phish if total_phish > 0 else 0,
        "VRAM Peak (GB)": torch.cuda.max_memory_allocated() / (1024**3),
        "Latency (s/img)": sum(latencies) / len(latencies)
    }

#### Test 16-bit (Bfloat16)

In [ ]:
from unsloth import FastVisionModel
import torch
import gc

# Completely clean up all possible previous model references from VRAM
for var in ['model', 'processor', 'model_8bit', 'processor_8bit', 'base_model_8bit']:
    if var in globals():
        del globals()[var]

torch.cuda.empty_cache()
gc.collect()
torch.cuda.reset_peak_memory_stats()

print("Loading Fine-Tuned Model in 16-bit (Bfloat16)...")
# Load the model directly without CPU offloading
model_16bit, processor_16bit = FastVisionModel.from_pretrained(
    "/content/qwen-vl-phishing-lora-final",
    load_in_4bit = False,
    dtype = torch.bfloat16,
    device_map = "cuda" # Force loading entirely on GPU to avoid CPU-offload Flash Attention errors
)

print(f"\nEvaluating 16-bit model on {len(test_subset)} samples...")
metrics_16bit = evaluate_quantization(model_16bit, processor_16bit, test_subset)
print("16-bit Evaluation Complete!")

Loading Fine-Tuned Model in 16-bit (Bfloat16)...
==((====))==  Unsloth 2026.11.4: Fast Qwen2_5_Vl patching. Transformers: 5.17.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/57.7k [00:00<?, ?B/s]

Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Evaluating 16-bit model on 20 samples...


... (4 outputs left out)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Processed 15/20...


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Processed 20/20...
16-bit Evaluation Complete!


#### Test 8-bit
load the 8-bit base model, and then attach our LoRA adapter via `peft`.

In [ ]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel

# Clear VRAM from 16-bit run
del model_16bit
del processor_16bit
torch.cuda.empty_cache()
gc.collect()
torch.cuda.reset_peak_memory_stats()

print("Loading Fine-Tuned Model in 8-bit (BitsAndBytes)...")
quant_config = BitsAndBytesConfig(load_in_8bit=True)

# Load 8-bit Base Model
base_model_8bit = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    device_map="auto",
    quantization_config=quant_config,
)

# Attach our saved LoRA adapter
model_8bit = PeftModel.from_pretrained(base_model_8bit, "/content/qwen-vl-phishing-lora-final")
processor_8bit = AutoProcessor.from_pretrained("/content/qwen-vl-phishing-lora-final")

print(f"\nEvaluating 8-bit model on {len(test_subset)} samples...")
metrics_8bit = evaluate_quantization(model_8bit, processor_8bit, test_subset)
print("8-bit Evaluation Complete!")

Loading Fine-Tuned Model in 8-bit (BitsAndBytes)...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]


Evaluating 8-bit model on 20 samples...


Streaming output truncated to the last 5000 lines.


Processed 5/20...


Streaming output truncated to the last 5000 lines.


Processed 10/20...


Streaming output truncated to the last 5000 lines.


#### Test 4-bit
use the saved model

In [ ]:
from unsloth import FastVisionModel
import pandas as pd
import gc
import torch

# Clear VRAM from 8-bit run
if 'model_8bit' in globals(): del model_8bit
if 'processor_8bit' in globals(): del processor_8bit
if 'base_model_8bit' in globals(): del base_model_8bit
torch.cuda.empty_cache()
gc.collect()
torch.cuda.reset_peak_memory_stats()

print("Loading Fine-Tuned Model in 4-bit (Unsloth)...")
model_4bit, processor_4bit = FastVisionModel.from_pretrained(
    "/content/qwen-vl-phishing-lora-final",
    load_in_4bit = True,
)

print(f"\nEvaluating 4-bit model on {len(test_subset)} samples...")
metrics_4bit = evaluate_quantization(model_4bit, processor_4bit, test_subset)
print("4-bit Evaluation Complete!\n")

# Create a comparison DataFrame
df_compare = pd.DataFrame([
    metrics_16bit,
    metrics_8bit,
    metrics_4bit
], index=["16-bit (BF16)", "8-bit (BitsAndBytes)", "4-bit (BitsAndBytes/Unsloth)"])

# Highlight the best values (max for accuracy/precision/etc, min for FNR/VRAM/Latency)
styled_df = df_compare.style.highlight_max(subset=["Accuracy", "Precision", "Recall", "F1-Score"], color='lightgreen') \
                            .highlight_min(subset=["FNR", "VRAM Peak (GB)", "Latency (s/img)"], color='lightgreen')

print("=== Quantization Comparison ===")
display(styled_df)


### 6. Interactive Multi-Precision Webpage Inference
Use this cell to evaluate your own image path across all three fine-tuned model versions (16-bit, 8-bit, and 4-bit) and compare their predictions, latency, and VRAM footprints.

In [ ]:
import os
import gc
import torch
import time
import json
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel
from unsloth import FastVisionModel
from qwen_vl_utils import process_vision_info

# @markdown ### Provide the image path to analyze
IMAGE_PATH = "/content/kaggle_phishing_dataset/screenshots/phishing_site_1/astuin.org_48.png" # @param {type:"string"}
USER_PROMPT = "Analyze this webpage screenshot. Identify the target brand, list suspicious visual or textual cues, and classify whether it is phishing in JSON format." # @param {type:"string"}

if not os.path.exists(IMAGE_PATH):
    print(f"Error: The file '{IMAGE_PATH}' does not exist! Please check the path.")
else:
    results = {}

    # Helper to run unified inference
    def run_inference(model, processor, path, prompt):
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": path},
                {"type": "text", "text": prompt}
            ]
        }]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to("cuda")

        start = time.time()
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=128, temperature=0.0, do_sample=False)
        latency = time.time() - start

        generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
        output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        vram = torch.cuda.max_memory_allocated() / (1024**3)
        return output_text, latency, vram

    # --- 1. 16-BIT INFERENCE ---
    print("\n--- Running 16-bit (BF16) Fine-Tuned Model ---")
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()

    model_16bit, processor_16bit = FastVisionModel.from_pretrained(
        "/content/qwen-vl-phishing-lora-final",
        load_in_4bit=False,
        dtype=torch.bfloat16,
        device_map="cuda"
    )
    if hasattr(model_16bit, "for_inference"): model_16bit.for_inference()

    output, latency, vram = run_inference(model_16bit, processor_16bit, IMAGE_PATH, USER_PROMPT)
    results["16-bit (BF16)"] = {"output": output, "latency": latency, "vram": vram}

    del model_16bit, processor_16bit

    # --- 2. 8-BIT INFERENCE ---
    print("\n--- Running 8-bit (BitsAndBytes) Fine-Tuned Model ---")
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()

    quant_config = BitsAndBytesConfig(load_in_8bit=True)
    base_model_8bit = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        "Qwen/Qwen2.5-VL-7B-Instruct",
        device_map="auto",
        quantization_config=quant_config,
    )
    model_8bit = PeftModel.from_pretrained(base_model_8bit, "/content/qwen-vl-phishing-lora-final")
    processor_8bit = AutoProcessor.from_pretrained("/content/qwen-vl-phishing-lora-final")
    model_8bit.eval()

    output, latency, vram = run_inference(model_8bit, processor_8bit, IMAGE_PATH, USER_PROMPT)
    results["8-bit (BitsAndBytes)"] = {"output": output, "latency": latency, "vram": vram}

    del model_8bit, processor_8bit, base_model_8bit

    # --- 3. 4-BIT INFERENCE ---
    print("\n--- Running 4-bit (Unsloth) Fine-Tuned Model ---")
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()

    model_4bit, processor_4bit = FastVisionModel.from_pretrained(
        "/content/qwen-vl-phishing-lora-final",
        load_in_4bit=True,
    )
    if hasattr(model_4bit, "for_inference"): model_4bit.for_inference()

    output, latency, vram = run_inference(model_4bit, processor_4bit, IMAGE_PATH, USER_PROMPT)
    results["4-bit (Unsloth)"] = {"output": output, "latency": latency, "vram": vram}

    del model_4bit, processor_4bit
    torch.cuda.empty_cache()
    gc.collect()

    # --- PRINT COMPARATIVE RESULTS ---
    print("\n" + "="*60)
    print("                  COMPARISON REPORT")
    print("="*60)
    for precision, data in results.items():
        print(f"\nPrecision: {precision}")
        print(f"Latency:   {data['latency']:.2f} seconds")
        print(f"Peak VRAM: {data['vram']:.2f} GB")
        print(f"Prediction Output:\n{data['output']}")
        print("-"*60)